In [1]:
import pandas as pd 
import duckdb 

In [3]:
conn=duckdb.connect(r"D:\SEC.gov project\database\credit_risk.db")

In [4]:
from IPython.core.magic import register_cell_magic

@register_cell_magic
def dsql(line, cell):
    df = conn.sql(cell).df()
    var_name = line.strip()
    if var_name:
        globals()[var_name] = df
    return df

In [6]:
%%dsql
SELECT 
    MAX(period_end) AS max_period_end,  
    MIN(period_end) AS min_period_end,
FROM gold.financial_metrics


,max_period_end,min_period_end
0,2025-11-30,2014-12-31


In [7]:
conn.sql("""

CREATE OR REPLACE TABLE gold.dim_date AS

WITH date_range AS (

    SELECT
        MIN(period_end) AS min_date,
        MAX(period_end) AS max_date

    FROM gold.financial_metrics

)

SELECT

    CAST(strftime(d, '%Y%m%d') AS INTEGER) AS date_key,

    d AS date,

    YEAR(d) AS year,

    CONCAT('Q', QUARTER(d)) AS quarter,

    QUARTER(d) AS quarter_number,

    MONTH(d) AS month,

    MONTH(d) AS month_number,

    strftime(d, '%B') AS month_name,

    CONCAT(
        YEAR(d),
        '-Q',
        QUARTER(d)
    ) AS year_quarter

FROM date_range,

     generate_series(
         min_date,
         max_date,
         INTERVAL '1 day'
     ) AS t(d)

""")

In [15]:
conn.sql("SHOW ALL TABLES").df()

,database,schema,name,column_names,column_types,temporary
0,credit_risk,clean,num,"[adsh, tag, version, ddate, qtrs, uom, value]","[VARCHAR, VARCHAR, VARCHAR, DATE, INTEGER, VAR...",False
1,credit_risk,clean,sub,"[adsh, cik, name, sic, fye, form, period, fy, ...","[VARCHAR, VARCHAR, VARCHAR, INTEGER, INTEGER, ...",False
2,credit_risk,clean,tag,"[tag, version, custom, abstract, datatype, ior...","[VARCHAR, VARCHAR, BOOLEAN, BOOLEAN, VARCHAR, ...",False
3,credit_risk,gold,dim_date,"[date_key, date, year, quarter, quarter_number...","[INTEGER, TIMESTAMP, BIGINT, VARCHAR, BIGINT, ...",False
4,credit_risk,gold,dim_filing,"[accession_number, company_id, company_name, i...","[VARCHAR, VARCHAR, VARCHAR, INTEGER, VARCHAR, ...",False
5,credit_risk,gold,fact_financial_metrics,"[accession_number, period_end, current_ratio, ...","[VARCHAR, DATE, DOUBLE, DOUBLE, DOUBLE, DOUBLE...",False
6,credit_risk,main,num,"[adsh, tag, version, ddate, qtrs, uom, dimh, i...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False
7,credit_risk,main,processed_quarters,"[quarter, status, processed_at, sub_rows, num_...","[VARCHAR, VARCHAR, TIMESTAMP, BIGINT, BIGINT, ...",False
8,credit_risk,main,sic_lookup,"[sic, industry_name, industry_group]","[VARCHAR, VARCHAR, VARCHAR]",False
9,credit_risk,main,sub,"[adsh, cik, name, sic, changed, afs, wksi, fye...","[VARCHAR, VARCHAR, VARCHAR, VARCHAR, VARCHAR, ...",False


In [14]:
conn.sql("""
ALTER TABLE gold.financial_metrics
RENAME TO fact_financial_metrics;""")

In [ ]:
%%dsql 
SELECT 
    industry_group,
    industry_name,
    COUNT(*) AS total_observations
FROM gold.dim_filing
JOIN gold.fact_financial_metrics USING (accession_number)
GROUP BY 1, 2
ORDER BY 1, 2

,industry_group,industry_name,total_observations
0,Industrial and Commercial Machinery and Comput...,CONSTRUCTION MACHINERY & EQUIP,725
1,Industrial and Commercial Machinery and Comput...,ENGINES & TURBINES,988
2,Industrial and Commercial Machinery and Comput...,FARM MACHINERY & EQUIPMENT,638
3,Industrial and Commercial Machinery and Comput...,"INDUSTRIAL TRUCKS, TRACTORS, TRAILERS & STACKERS",228
4,Transportation Equipment,MOTOR HOMES,180
5,Transportation Equipment,MOTOR VEHICLE PARTS & ACCESSORIES,4749
6,Transportation Equipment,MOTOR VEHICLES & PASSENGER CAR BODIES,1963
7,Transportation Equipment,TRUCK & BUS BODIES,425
8,Transportation Equipment,TRUCK TRAILERS,189


In [21]:
conn.close

<bound method pybind11_detail_function_record_v1_msvc_mt_mscver1944.close of <_duckdb.DuckDBPyConnection object at 0x000001B0F4D437F0>>